In [2]:
import pandas as pd
import numpy as np
import re
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout

In [3]:
from google.colab import files
uploaded = files.upload()

Saving training.csv to training.csv


In [4]:
# Load dataset
columns = ["target", "id", "date", "query_flag", "user", "tweet_text"]

df = pd.read_csv(
    "training.csv",
    encoding="latin-1",
    header=None,
    names=columns
)

df = df[["target", "tweet_text"]]

# Convert labels: 0 = negative, 4 = positive
df["target"] = df["target"].replace(4, 1)

df.head()

,target,tweet_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["tweet_text"] = df["tweet_text"].apply(clean_text)

In [6]:
X = df["tweet_text"].values
y = df["target"].values

# 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 10% validation, 10% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [7]:
max_words = 50000
max_len = 50

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post")

In [8]:
rnn_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
start_time = time.time()

rnn_history = rnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=128
)

rnn_training_time = time.time() - start_time

Epoch 1/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 1402s 140ms/step - accuracy: 0.6650 - loss: 0.6176 - val_accuracy: 0.6144 - val_loss: 0.6676
Epoch 2/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 1401s 140ms/step - accuracy: 0.7599 - loss: 0.5046 - val_accuracy: 0.7891 - val_loss: 0.4612
Epoch 3/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 1376s 137ms/step - accuracy: 0.8015 - loss: 0.4436 - val_accuracy: 0.7828 - val_loss: 0.4672
Epoch 4/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 1385s 138ms/step - accuracy: 0.8069 - loss: 0.4360 - val_accuracy: 0.7936 - val_loss: 0.4618
Epoch 5/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 1403s 140ms/step - accuracy: 0.8102 - loss: 0.4298 - val_accuracy: 0.7936 - val_loss: 0.4620


In [10]:
rnn_loss, rnn_accuracy = rnn_model.evaluate(X_test_pad, y_test)

print("RNN Test Loss:", rnn_loss)
print("RNN Test Accuracy:", rnn_accuracy)
print("RNN Training Time:", rnn_training_time)

5000/5000 ━━━━━━━━━━━━━━━━━━━━ 40s 8ms/step - accuracy: 0.7943 - loss: 0.4610
RNN Test Loss: 0.46103549003601074
RNN Test Accuracy: 0.7942875027656555
RNN Training Time: 6967.114444971085


In [11]:
lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
start_time = time.time()

lstm_history = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=128
)

lstm_training_time = time.time() - start_time

Epoch 1/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 2300s 230ms/step - accuracy: 0.7893 - loss: 0.4389 - val_accuracy: 0.8203 - val_loss: 0.3993
Epoch 2/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 2286s 229ms/step - accuracy: 0.8359 - loss: 0.3694 - val_accuracy: 0.8285 - val_loss: 0.3834
Epoch 3/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 2268s 227ms/step - accuracy: 0.8527 - loss: 0.3371 - val_accuracy: 0.8267 - val_loss: 0.3873
Epoch 4/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 2296s 230ms/step - accuracy: 0.8688 - loss: 0.3053 - val_accuracy: 0.8241 - val_loss: 0.4047
Epoch 5/5
10000/10000 ━━━━━━━━━━━━━━━━━━━━ 2276s 228ms/step - accuracy: 0.8845 - loss: 0.2735 - val_accuracy: 0.8205 - val_loss: 0.4287


In [13]:
lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test_pad, y_test)

print("LSTM Test Loss:", lstm_loss)
print("LSTM Test Accuracy:", lstm_accuracy)
print("LSTM Training Time:", lstm_training_time)

5000/5000 ━━━━━━━━━━━━━━━━━━━━ 129s 26ms/step - accuracy: 0.8200 - loss: 0.4275
LSTM Test Loss: 0.4275355339050293
LSTM Test Accuracy: 0.8200374841690063
LSTM Training Time: 11427.70489692688


In [14]:
results = pd.DataFrame({
    "Model": ["Simple RNN", "LSTM"],
    "Training Time (seconds)": [rnn_training_time, lstm_training_time],
    "Test Loss": [rnn_loss, lstm_loss],
    "Test Accuracy": [rnn_accuracy, lstm_accuracy]
})

results

,Model,Training Time (seconds),Test Loss,Test Accuracy
0,Simple RNN,6967.114445,0.461035,0.794288
1,LSTM,11427.704897,0.427536,0.820037
